In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os, sys
utils_path = os.path.abspath(os.path.join(os.getcwd(), '..', '2_data_analysis', 'utils'))
sys.path.append(utils_path)
import plot_style

C_AGR   = '#C0392B'
C_NOAGR = '#2C6E8A'

## Build new columns

---

### Conflict Panel

In [7]:
df_panel = pd.read_csv('../../data/output/conflict_level/conflict_panel.csv', low_memory=False)

print(f"Total conflicts: {len(df_panel['conflict_id'].unique())}")
print(f"With agreement:   {df_panel.loc[df_panel['ever_agreement']==1]['conflict_id'].nunique()}")
print(f"\t agreement during conflict: {df_panel.loc[(df_panel['ever_agreement']==1) & (df_panel['first_agreement_date']>=df_panel['start_date_numeric']) & (df_panel['first_agreement_date']<=df_panel['end_date_numeric'])]['conflict_id'].nunique()}")
print(f"\t agreement before start_date: {df_panel.loc[(df_panel['ever_agreement']==1) & (df_panel['first_agreement_date']<df_panel['start_date_numeric'])]['conflict_id'].nunique()}")
print(f"\t agreement after end_date: {df_panel.loc[(df_panel['ever_agreement']==1) & (df_panel['first_agreement_date']>df_panel['end_date_numeric'])]['conflict_id'].nunique()}")
print(f"Without agreement: {df_panel.loc[df_panel['ever_agreement']==0]['conflict_id'].nunique()}")

Total conflicts: 201
With agreement:   104
	 agreement during conflict: 82
	 agreement before start_date: 11
	 agreement after end_date: 11
Without agreement: 97


### Incompatibility type

Territorial conflicts have harder-to-divide stakes.

In [8]:
#merge the incompatibility column from UCDP/PRIO
df_prio = pd.read_csv('../../data/input/ucdp/UcdpPrioConflict_v25_1.csv', low_memory=False)
df_panel = df_panel.merge(df_prio[['conflict_id', 'year', 'incompatibility', 'type_of_conflict']], left_on = ['conflict_id', 'year'], right_on = ['conflict_id', 'year'], how = 'left')
df_panel['incompatibility'] = df_panel.groupby('conflict_id')['incompatibility'].ffill().bfill().astype(int)
type_of_conflict_map = {1: "extrasystemic", 2: "intrastate", 3: "interstate", 4: "internationalized_intrastate"}
df_panel['type_of_conflict'] = df_panel.groupby('conflict_id')['type_of_conflict'].ffill().bfill().map(type_of_conflict_map)
df_panel[['conflict_id', 'year_mo', 'incompatibility', 'type_of_conflict']]

,conflict_id,year_mo,incompatibility,type_of_conflict
0,205,1989-01,1,interstate
1,205,1989-02,1,interstate
2,205,1989-03,1,interstate
3,205,1989-04,1,interstate
4,205,1989-05,1,interstate
...,...,...,...,...
86827,16379,2024-08,1,interstate
86828,16379,2024-09,1,interstate
86829,16379,2024-10,1,interstate
86830,16379,2024-11,1,interstate


### Termination outcome

In [9]:
from functions.build_termination_outcome import build_termination_outcome

df_panel = build_termination_outcome(
    df_panel,
    "../../data/input/ucdp/UCDPConflictTerminationDataset_v4_2024_Conflict.csv"
)

df_panel[['conflict_id', 'year_mo', 'agree_timing', 'termination_outcome_label', 'cause_label', 'effective_end_date']]

,conflict_id,year_mo,agree_timing,termination_outcome_label,cause_label,effective_end_date
0,205,1989-01,never_signed,Low activity,fade,2018-10
1,205,1989-02,never_signed,Low activity,fade,2018-10
2,205,1989-03,never_signed,Low activity,fade,2018-10
3,205,1989-04,never_signed,Low activity,fade,2018-10
4,205,1989-05,never_signed,Low activity,fade,2018-10
...,...,...,...,...,...,...
86827,16379,2024-08,never_signed,ongoing,censored,2024-12
86828,16379,2024-09,never_signed,ongoing,censored,2024-12
86829,16379,2024-10,never_signed,ongoing,censored,2024-12
86830,16379,2024-11,never_signed,ongoing,censored,2024-12


In [10]:
df_panel.groupby(['cause_label', 'termination_outcome_label'], dropna=False)['conflict_id'].nunique().reset_index(name='n_conflicts').sort_values('n_conflicts', ascending=False)

,cause_label,termination_outcome_label,n_conflicts
0,censored,ongoing,39
1,fade,Low activity,31
8,first_agreement,ongoing,31
10,not at risk,Victory (govt),20
4,first_agreement,Low activity,19
5,first_agreement,Peace agreement,18
3,first_agreement,Ceasefire,16
6,first_agreement,Victory (govt),7
9,not at risk,Actor ceases to exist,7
11,not at risk,Victory (rebels),6


In [11]:
df_panel[df_panel['termination_outcome_label'].isin(['Peace agreement','Ceasefire'])].groupby('conflict_id').agg(
    first_agreement_date=('first_agreement_date', 'first'),
    effective_end_date=('effective_end_date', 'first'),
    start_date=('start_date', 'first'),
    end_date=('end_date', 'first'),
    termination_outcome_label=('termination_outcome_label', 'first'),
    cause_label=('cause_label', 'first'),
    ever_agreement=('ever_agreement', 'first'),
    country = ('country', 'first'))

,first_agreement_date,effective_end_date,start_date,end_date,termination_outcome_label,cause_label,ever_agreement,country
conflict_id,,,,,,,,
218,28,2023-11,1989-01,2023-11,Ceasefire,first_agreement,1,India
233,15,1996-11,1989-01,1996-11,Peace agreement,first_agreement,1,Guatemala
251,103,2023-09,1992-05,2023-09,Ceasefire,first_agreement,1,India
269,203,2009-12,1996-02,2009-12,Peace agreement,first_agreement,1,Nepal
274,24251,2020-12,2020-06,2020-08,Ceasefire,first_agreement,1,India
292,24131,2010-12,1989-01,2023-09,Ceasefire,first_agreement,1,Peru
298,29,1989-05,1989-02,1989-05,Peace agreement,first_agreement,1,South Africa
315,60,2019-04,1989-01,2019-04,Ceasefire,first_agreement,1,United Kingdom
316,16,1992-01,1989-01,1992-01,Peace agreement,first_agreement,1,El Salvador


In [6]:
# incompatibility: 1 = government control, 2 = territorial control (Fearon 1995: harder to divide)
df_panel['territorial_incompatibility'] = (df_panel['incompatibility'] == 2).astype(int)

cl = df_panel.drop_duplicates('conflict_id')
print(f"Territorial incompatibility: {cl['territorial_incompatibility'].sum()} of {len(cl)} conflicts")

Territorial incompatibility: 74 of 201 conflicts


## Information asymmetry proxies

---

Past fighting between the same parties reduces uncertainty about relative strength. `experience_total` is the discounted count of prior dyadic conflicts (δ = 0.95/yr). `high_ia_bin = 1` when `experience_total == 0` at onset.

### Experience fighting

In [62]:
from functions.build_experience_compound import build_experience_expanded

df_panel = build_experience_expanded(
    df_panel,
    "../../data/input/ucdp/Dyadic_v25_1.csv",
    delta=0.95,
    weight_direct=1.0,
    weight_government=0.5,
)

df_panel[['conflict_id', 'year_mo', 'experience_total', 'exp_direct', 'exp_government']]

Experience computed for 201 conflicts:
  exp_direct = 0 in 133/201 conflicts (66%)
  experience_total = 0 in 77/201 conflicts (38%)
  (fewer zeros = more variation for IA proxy)


,conflict_id,year_mo,experience_total,exp_direct,exp_government
0,205,1989-01,12.164151,8.270352,7.787599
1,205,1989-02,12.164151,8.270352,7.787599
2,205,1989-03,12.164151,8.270352,7.787599
3,205,1989-04,12.164151,8.270352,7.787599
4,205,1989-05,12.164151,8.270352,7.787599
...,...,...,...,...,...
86827,16379,2024-08,7.250709,0.000000,14.501418
86828,16379,2024-09,7.250709,0.000000,14.501418
86829,16379,2024-10,7.250709,0.000000,14.501418
86830,16379,2024-11,7.250709,0.000000,14.501418


## Commitment proxies

---

Onset / time-invariant proxies following Fearon (1995) and Feedback §3. Dynamic measures (factions, external support) are Exercise B only and excluded here.

### Ethnic fractionalization (Fearon 2003)

In [63]:
# Fearon & Laitin (2003) ethnic fractionalization via QoG dataset.
BASE = '../../data/input/fearon_ethnic_fractionalization/'

def load_fe(fname, varname):
    df = pd.read_csv(BASE + fname, index_col=0, sep=';', decimal=',')
    return df.groupby('ccodealp')[varname].first().reset_index()

fe_vars = {
    'fe_etfra':   'qogdata_08_06_2026_fe_etfra.csv',
    'fe_cultdiv': 'qogdata_08_06_2026_fe_cultdiv.csv',
}

# Regional mean lookup for imputation of missing countries
_fe_raw   = pd.read_csv(BASE + 'qogdata_08_06_2026_fe_etfra.csv', index_col=0, sep=';', decimal=',')
_reg_lkp  = df_panel[['isocode', 'region']].drop_duplicates()
_fe_u     = _fe_raw.groupby('ccodealp')['fe_etfra'].first().reset_index()
_reg_mean = (
    _fe_u.merge(_reg_lkp, left_on='ccodealp', right_on='isocode', how='left')
    .groupby('region')['fe_etfra'].mean().to_dict()
)
_val    = lambda c: _fe_raw[_fe_raw['ccodealp']==c]['fe_etfra'].iloc[0] \
                   if c in _fe_raw['ccodealp'].values else None
_manual = {k: v for k, v in {'SRB': _val('SCG') or _val('YUG'), 'SSD': _val('SDN')}.items()
           if v is not None}
print(f"Manual imputation cases: {_manual}")

Manual imputation cases: {'SSD': np.float64(0.71)}


### UN Peacekeeping at onset — `pko_onset` (Geo-PKO v2.3)

UN peacekeeping presence at conflict onset is a proxy for prior international engagement and third-party enforcement capacity (Walter 1997). A pre-existing PKO signals that external actors were already involved before the conflict, which may lower commitment problems for rebel demobilization.

**Source:** Geocoded Peacekeeping Operations Dataset (Geo-PKO v2.3, Uppsala University), location × month panel, 1994–2024.

**Coverage limitation:** Geo-PKO starts in 1994. Of the 201 conflicts in the panel, **98 have onset before 1994** (panel runs from 1989). For all pre-1994 conflicts `pko_onset = 0` by construction — not because there was no PKO but because data is unavailable. 

In [69]:
pko_raw.groupby(['country', 'year_mo']).size().reset_index(name='_n').assign(pko_month=1)

,country,year_mo,_n,pko_month
0,Afghanistan,1996-12,1,1
1,Afghanistan,1997-01,1,1
2,Afghanistan,1997-02,1,1
3,Afghanistan,1997-03,1,1
4,Afghanistan,1997-04,1,1
...,...,...,...,...
7242,Zimbabwe,2002-09,1,1
7243,Zimbabwe,2002-10,1,1
7244,Zimbabwe,2002-11,1,1
7245,Zimbabwe,2002-12,1,1


In [64]:
pko_raw = pd.read_csv('../../data/input/ucdp/Geo_PKO_v.2.3_location_month.csv', low_memory=False)

_PKO_NAME_FIX = {
    'Bosnia and Herzegovina': 'Bosnia-Herzegovina',
    'DRC':                    'DR Congo (Zaire)',
}
pko_raw['country'] = pko_raw['country'].replace(_PKO_NAME_FIX)

# Country × year_mo presence indicator (any row = PKO deployed)
pko_raw['year_mo'] = (
    pko_raw['year'].astype(str) + '-' +
    pko_raw['month'].astype(str).str.zfill(2)
)
pko_monthly = (
    pko_raw
    .groupby(['country', 'year_mo'])
    .size()
    .reset_index(name='_n')
    .assign(pko_month=1)
    [['country', 'year_mo', 'pko_month']]
)

cl_onset = (
    df_panel
    .drop_duplicates('conflict_id')
    [['conflict_id', 'country', 'start_date']]
    .copy()
)

# ── pko_onset ─────────────────────────────────────────────────────────────────
# PKO present in the country in the conflict onset month (Walter 1997:
# pre-existing third-party enforcement capacity). Geo-PKO starts 1994, so all
# 98 pre-1994 conflicts get pko_onset = 0 by data limitation, not by design.
pko_onset_df = (
    cl_onset
    .merge(
        pko_monthly,
        left_on=['country', 'start_date'],
        right_on=['country', 'year_mo'],
        how='left',
    )
    [['conflict_id', 'pko_month']]
    .fillna({'pko_month': 0})
    .rename(columns={'pko_month': 'pko_onset'})
    .assign(pko_onset=lambda d: d['pko_onset'].astype(int))
)

# ── pko_ever ──────────────────────────────────────────────────────────────────
# PKO present in the country at ANY month during the conflict spell
# (start_date to end_date). Measures international engagement/attention,
# not prior enforcement capacity. Interpretation: international actors
# deemed the conflict worth a peacekeeping mission at some point.
# Note: this is endogenous to conflict intensity — PKOs are deployed in
# response to violence, not randomly. Use as correlate, not causal proxy.
df_with_pko = df_panel.merge(
    pko_monthly,
    left_on=['country', 'year_mo'],
    right_on=['country', 'year_mo'],
    how='left',
)
df_with_pko['pko_month'] = df_with_pko['pko_month'].fillna(0).astype(int)

spell_mask = (
    (df_with_pko['year_mo'] >= df_with_pko['start_date']) &
    (df_with_pko['year_mo'] <= df_with_pko['end_date'])
)
pko_ever_df = (
    df_with_pko[spell_mask]
    .groupby('conflict_id')['pko_month']
    .max()
    .reset_index()
    .rename(columns={'pko_month': 'pko_ever'})
    .assign(pko_ever=lambda d: d['pko_ever'].astype(int))
)

# Merge both into df_panel
df_panel = df_panel.drop(columns=['pko_onset', 'pko_ever'], errors='ignore')
df_panel = df_panel.merge(pko_onset_df, on='conflict_id', how='left')
df_panel = df_panel.merge(pko_ever_df, on='conflict_id', how='left')

# Summary
cl_check  = df_panel.drop_duplicates('conflict_id')
n_pre94   = (cl_check['start_date'].str[:4].astype(int) < 1994).sum()
print(f"pko_onset = 1 (PKO at onset month, post-1994 only): {(cl_check['pko_onset']==1).sum()} conflicts")
print(f"pko_onset = 0 (no PKO or pre-1994 data gap):        {(cl_check['pko_onset']==0).sum()} conflicts")
print(f"  of which pre-1994 onset (data unavailable):       {n_pre94} conflicts")
print()
print(f"pko_ever  = 1 (PKO at any point during spell):      {(cl_check['pko_ever']==1).sum()} conflicts")
print(f"pko_ever  = 0 (no PKO deployment during spell):     {(cl_check['pko_ever']==0).sum()} conflicts")

pko_onset = 1 (PKO at onset month, post-1994 only): 25 conflicts
pko_onset = 0 (no PKO or pre-1994 data gap):        176 conflicts
  of which pre-1994 onset (data unavailable):       98 conflicts

pko_ever  = 1 (PKO at any point during spell):      73 conflicts
pko_ever  = 0 (no PKO deployment during spell):     128 conflicts


## **Spell Dataset**

---

A **survival dataset** (or spell dataset) records each conflict's history as a sequence of periods until an event occurs or the observation ends without one. Each row is one **conflict × period** observation; the event of interest is the signing of the **first peace agreement**.

Two filtering steps define the spell:

1. **Trim to the active conflict window** — keep only observations between `start_date` and `end_date` (UCDP conflict onset and last recorded activity).
2. **Censor at first agreement** — drop all observations after the first agreement. Each conflict's spell ends either at the event quarter (`is_first_agreement = 1`) or at the last observed period (right-censored, `ever_agreement = 0`).

This structure supports a **discrete-time hazard model**: at each period, the conflict either transitions to an agreement or remains at risk. The primary estimation dataset is aggregated at the **quarterly level** (`spell_q`), which is the unit of analysis for the ClogLog model.

In [12]:
# Trim to active conflict window using effective_end_date.
# For non-signers where GED activity extends beyond the UCDP termination date,
# effective_end_date = c_ependdate (spell truncated to true exit).
# For all other conflicts, effective_end_date = end_date (GED).
df_panel_capped = df_panel[
    (df_panel['year_mo'] >= df_panel['start_date']) &
    (df_panel['year_mo'] <= df_panel['effective_end_date'])
]

# Keep rows up to first agreement for signers; full corrected spell for non-signers.
pre_treatment = df_panel_capped[
    (df_panel_capped["ever_agreement"] == 0) |
    (df_panel_capped["year_mo_numeric"] <= df_panel_capped["first_agreement_date"])
].copy()

pre_treatment = pre_treatment.sort_values(["conflict_id", "year_mo_numeric"])
print(f"Spell: {pre_treatment['conflict_id'].nunique()} conflicts, {len(pre_treatment):,} rows")
print()
print("cause_label distribution (conflict level):")
print(pre_treatment.drop_duplicates('conflict_id')['cause_label'].value_counts().sort_index().to_string())
print()
print("agree_timing distribution:")
print(pre_treatment.drop_duplicates('conflict_id')['agree_timing'].value_counts().sort_index().to_string())

Spell: 201 conflicts, 18,235 rows

cause_label distribution (conflict level):
cause_label
censored           39
fade               31
first_agreement    98
not at risk        33

agree_timing distribution:
agree_timing
during_ged      95
never_signed    94
pre_ged          9
ucdp_only        3


In [13]:
pre_treatment['is_first_agreement'] = (
    (pre_treatment['ever_agreement'] == 1) &
    (pre_treatment['year_mo_numeric'] == pre_treatment['first_agreement_date'])
).astype(int)

print("is_first_agreement per conflict:")
print(pre_treatment.groupby('conflict_id')['is_first_agreement'].sum().value_counts())

is_first_agreement per conflict:
is_first_agreement
0    119
1     82
Name: count, dtype: int64


In [14]:
pre_treatment['log_conflict_age'] = np.log1p(pre_treatment['conflict_age'])

## **Quarterly Aggregation**

---

The monthly spell is collapsed to **quarters** (`spell_q`), the unit of analysis for the discrete-time ClogLog model. Time-varying covariates (fatalities, event counts, rolling IA measures, CP index) are aggregated within each conflict-quarter; time-invariant covariates (experience, initial IA) are carried forward from the first observation. Quarterly IA and CP variables are also recomputed at this frequency.

In [ ]:
pre_treatment["yq"]           = pd.to_datetime(pre_treatment["year_mo"]).dt.to_period("Q")
pre_treatment['start_date_q'] = pd.to_datetime(pre_treatment['start_date']).dt.to_period("Q").dt.to_timestamp()
pre_treatment['end_date_q']   = pd.to_datetime(pre_treatment['end_date']).dt.to_period("Q").dt.to_timestamp()

_AGG = {
    "best":                        "sum",
    "n_events":                    "sum",
    "is_first_agreement":          "max",
    "ever_agreement":              "max",
    "cause_label":                 "first",
    "termination_outcome_label":   "first",
    "conflict_age":                "max",
    "isocode":                     "first",
    "country":                     "first",
    "region":                      "first",
    "type_of_conflict":            "first",
    "incompatibility":             "first",
    "territorial_incompatibility": "first",
    "experience_total":            "first",
    "pko_onset":                   "first",
    "pko_ever":                    "first",
    "start_date_q":                "first",
    "end_date_q":                  "first",
}

spell_q = (
    pre_treatment.groupby(["conflict_id", "yq"])
    .agg(_AGG)
    .reset_index()
    .sort_values(["conflict_id", "yq"])
    .reset_index(drop=True)
)

spell_q["conflict_age_q"]     = spell_q.groupby("conflict_id").cumcount() + 1
spell_q["log_conflict_age_q"] = np.log1p(spell_q["conflict_age_q"])
spell_q["high_ia_bin"]        = (spell_q["experience_total"] == 0).astype(int)

print(f"Quarterly spell: {len(spell_q):,} rows, {spell_q['conflict_id'].nunique()} conflicts")
print(f"Events (is_first_agreement==1): {spell_q['is_first_agreement'].sum()}")
print(f"\ncause_label distribution:")
print(spell_q.drop_duplicates('conflict_id')['cause_label'].value_counts().sort_index().to_string())
print(f"\nhigh_ia_bin (no prior experience): {spell_q['high_ia_bin'].mean():.1%} of quarters")

In [ ]:
for varname, fname in fe_vars.items():
    static = load_fe(fname, varname)
    spell_q = spell_q.drop(columns=[varname, 'ccodealp'], errors='ignore')
    spell_q = spell_q.merge(static, left_on='isocode', right_on='ccodealp', how='left')
    spell_q = spell_q.drop(columns='ccodealp')

def _impute_fe(row):
    if pd.notna(row['fe_etfra']): return row['fe_etfra']
    if row['isocode'] in _manual:  return _manual[row['isocode']]
    return _reg_mean.get(row['region'], None)

spell_q['fe_etfra_imp']      = spell_q.apply(_impute_fe, axis=1)
spell_q['high_fe_etfra_bin'] = (spell_q['fe_etfra_imp'] > 0.5).astype(int)

print(f"fe_etfra_imp NaN: {spell_q['fe_etfra_imp'].isna().sum()}")
print(f"high_fe_etfra_bin (EF > 0.5): {spell_q['high_fe_etfra_bin'].mean():.1%} of quarters")

In [ ]:
_export_cols = [
    'conflict_id', 'yq', 'start_date_q', 'end_date_q',
    # outcome / competing risks
    'is_first_agreement', 'ever_agreement',
    'cause_label', 'termination_outcome_label',
    # baseline hazard
    'conflict_age_q', 'log_conflict_age_q',
    # raw intensity
    'best', 'n_events',
    # information asymmetry (onset)
    'experience_total', 'high_ia_bin',
    # commitment proxies (onset)
    'incompatibility', 'territorial_incompatibility',
    'fe_etfra_imp', 'high_fe_etfra_bin', 'fe_cultdiv',
    # pko_onset: Walter (1997) prior enforcement capacity
    #   → 0 for all 98 pre-1994 conflicts (data gap, not absence)
    #   → use only on post-1994 subsample for clean identification
    'pko_onset',
    # pko_ever: international engagement at any point during spell
    #   → broader coverage (captures pre-1994 conflicts if PKO arrived later)
    #   → endogenous to conflict intensity; use as correlate, not causal proxy
    'pko_ever',
    # identifiers
    'country', 'region', 'isocode', 'type_of_conflict',
]
_export_cols = list(dict.fromkeys(_export_cols))

_avail = [c for c in _export_cols if c in spell_q.columns]
_miss  = [c for c in _export_cols if c not in spell_q.columns]
if _miss:
    print(f"WARNING — not yet built: {_miss}")

spell_q[_avail].to_csv('../../data/output/conflict_level/spell_q.csv', index=False)
print(f"Saved {len(spell_q[_avail]):,} rows x {len(_avail)} cols → spell_q.csv")
print(f"Columns: {_avail}")

## **Export**

---

Save the quarterly spell dataset for use in `survival_analysis.ipynb`.
- `spell_q.csv` — loaded by the analysis notebook (Python)
- `spell_q.dta` — for the Stata ClogLog specification (`cloglog_hazard.do`)